# Review Manhwa

Cắt panel, đọc chữ, viết lời review tiếng Việt, đọc bằng **VieNeu-TTS v3 Turbo**, rồi ghép video. Giao diện web hiện ngay trong notebook.

Trước khi chạy: **Runtime → Change runtime type → T4 GPU**.

Repo private thì vào Colab → **Secrets**, thêm secret tên đúng `GITHUB_TOKEN` (quyền đọc nội dung repo) và bật **Notebook access**. Bảng Secrets hay cắt tên, `GITHUB_1` không phải tên code đọc.

Chỉ tải lên truyện bạn có quyền sử dụng. Tool không lấy chương từ website.

In [ ]:
import os
import shutil
import subprocess
import sys

ROOT = "/content/review-manhwa"
REPO = "https://github.com/hiepit127/review-manhwa.git"

def _read_secret(name):
    try:
        from google.colab import userdata
        return (userdata.get(name) or "").strip()
    except Exception:
        return os.environ.get(name, "").strip()

def _token():
    for name in ("GITHUB_TOKEN", "GITHUB_1", "GH_TOKEN", "GITHUB_PAT"):
        value = _read_secret(name)
        if value.lower().startswith("bearer "):
            value = value.split(None, 1)[1].strip()
        if value:
            print(f"Đã đọc secret {name}.")
            return value
    print("Không thấy secret GITHUB_TOKEN. Bảng Secrets đang cắt tên, hãy đặt đúng tên GITHUB_TOKEN.")
    return ""

def _redact(text, token):
    import re
    cleaned = text or ""
    if token:
        cleaned = cleaned.replace(token, "***")
    cleaned = re.sub(r"x-access-token:[^@\s]+", "x-access-token:***", cleaned)
    cleaned = re.sub(r"ghp_[A-Za-z0-9]+", "ghp_***", cleaned)
    cleaned = re.sub(r"github_pat_[A-Za-z0-9_]+", "github_pat_***", cleaned)
    return cleaned.strip()

def _clone(url, env):
    if os.path.isdir(ROOT):
        shutil.rmtree(ROOT)
    result = subprocess.run(
        ["git", "-c", "credential.helper=", "clone", "--depth", "1", url, ROOT],
        env=env,
        capture_output=True,
        text=True,
    )
    ok = result.returncode == 0 and os.path.exists(os.path.join(ROOT, "app.py"))
    if not ok:
        shutil.rmtree(ROOT, ignore_errors=True)
    return ok, (result.stderr or "") + (result.stdout or "")

if not os.path.exists("app.py"):
    if not os.path.exists(os.path.join(ROOT, "app.py")):
        env = os.environ.copy()
        env["GIT_TERMINAL_PROMPT"] = "0"
        token = _token()
        cloned = False
        detail = ""
        if token:
            from urllib.parse import quote
            authed = "https://x-access-token:" + quote(token, safe="") + "@github.com/hiepit127/review-manhwa.git"
            cloned, detail = _clone(authed, env)
            if cloned:
                subprocess.run(
                    ["git", "-C", ROOT, "remote", "set-url", "origin", REPO],
                    check=False,
                )
            else:
                print("GitHub từ chối token.")
                print(_redact(detail, token))
                print("Thử clone không dùng token.")
                cloned, detail = _clone(REPO, env)
        else:
            cloned, detail = _clone(REPO, env)
        if not cloned:
            raise SystemExit("Không clone được repo.\n" + _redact(detail, token))
    os.chdir(ROOT)

if os.path.isdir(".git"):
    pulled = subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True)
    if pulled.returncode == 0:
        print((pulled.stdout or "Đã cập nhật code.").strip())
    else:
        print("Không kéo được code mới. Dùng bản đang có trong máy.")

if not os.path.exists("requirements-colab.txt"):
    raise SystemExit(f"Không thấy requirements-colab.txt trong {os.getcwd()}")

if os.path.exists("/content"):
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg", "fonts-dejavu-core"], check=True)

keep = []
try:
    import importlib.metadata as metadata
    for name in ("torch", "torchvision", "numpy"):
        try:
            keep.append(f"{name}=={metadata.version(name)}")
        except metadata.PackageNotFoundError:
            pass
except Exception:
    pass
constraint = "/tmp/review-manhwa-keep.txt"
with open(constraint, "w", encoding="utf-8") as handle:
    handle.write("\n".join(keep) + "\n")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt", "-c", constraint],
    check=True,
)
subprocess.run([sys.executable, "app.py", "--doctor"], check=True)

try:
    import torch
    print("CUDA", torch.cuda.is_available(), torch.__version__)
    if not torch.cuda.is_available():
        print("Chưa thấy GPU. Chọn Runtime → Change runtime type → T4, rồi chạy lại ô này.")
except Exception as error:
    print(error)


In [ ]:
import os
import sys

sys.path.insert(0, os.getcwd())
from src.ports import close_ports

close_ports()


In [ ]:
import importlib
import os
import sys

sys.path.insert(0, os.getcwd())
import app
import src.ports

importlib.reload(src.ports)
importlib.reload(app)
app.launch()
